In [ ]:
import numpy as np
import pandas as pd
import requests
import csv
import sys
import os
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)
root_path = 'drive/My Drive/ECI Research/5. Datasets/2022 Fall - Data/Country_Code.csv'

Mounted at /content/drive/


#  Define Variables( Remember to create local directory before you start)

In [ ]:

reporter_code_list=pd.read_csv(root_path,encoding='ISO-8859-1')["Country Code"].values
year_list=np.array([2019,2020,2021])
PATH="drive/My Drive/ECI Research/5.Datasets/Data/raw_data/"

# Define helper functions

In [ ]:
# Function to web-scrap each country's export data from UN Comtrade and save to local directory by years.

def get_export(reporter_code,year):
    CSV_URL = 'https://comtrade.un.org/api/get?max=50000&type=C&freq=A&px=HS&ps='+str(year)+'&r='+str(reporter_code)+'&p=0&rg=2&cc=ALL&uitoken=<COMTRADE_UITOKEN>&fmt=csv'

    with requests.Session() as s:
        download = s.get(CSV_URL)
        decoded_content = download.content.decode('utf-8')
        cr = csv.reader(decoded_content.splitlines(), delimiter=',')
        my_list = list(cr)
        df=pd.DataFrame(np.array(my_list)).iloc[1:,[21,31]]
        df.columns=["commodity_code","dollar_value"]
        # only include six-digit commodity codes
        df=df[df["commodity_code"].apply(lambda x: len(str(x))==6)]
        if len(df)<20:
            raise ValueError
        df.to_csv(PATH+str(year)+"/"+str(reporter_code)+".csv")
    return df

In [ ]:
len(reporter_code_list)

294

In [ ]:
year=2022
reporter_code=76
CSV_URL = 'https://comtrade.un.org/api/get?max=50000&type=C&freq=A&px=HS&ps='+str(year)+'&r='+str(reporter_code)+'&p=0&rg=2&cc=ALL&uitoken=<COMTRADE_UITOKEN>&fmt=csv'

CSV_URL

'https://comtrade.un.org/api/get?max=50000&type=C&freq=A&px=HS&ps=2022&r=76&p=0&rg=2&cc=ALL&uitoken=<COMTRADE_UITOKEN>&fmt=csv'

In [ ]:

year=2022
reporter_code=76
CSV_URL = 'https://comtrade.un.org/api/get?max=50000&type=C&freq=A&px=HS&ps='+str(year)+'&r='+str(reporter_code)+'&p=0&rg=2&cc=ALL&uitoken=<COMTRADE_UITOKEN>&fmt=csv'

with requests.Session() as s:
  download = s.get(CSV_URL)
  decoded_content = download.content.decode('utf-8')
  cr = csv.reader(decoded_content.splitlines(), delimiter=',')
  my_list = list(cr)
  print(my_list)
  df=pd.DataFrame(np.array(my_list)).iloc[1:,[21,31]]
  df.columns=["commodity_code","dollar_value"]
        # only include six-digit commodity codes
  df=df[df["commodity_code"].apply(lambda x: len(str(x))==6)]
  if len(df)<20:
    raise ValueError
  df.to_csv(PATH+str(year)+"/"+str(reporter_code)+".csv")


SSLError: ignored

In [ ]:
# Function to read in all csvs from local, merge into a mega csv, and save to local.

def merge_all(PATH,year_list):
    for year in year_list:
        sub_path=PATH+str(year)
        print(sub_path)
        # fetch all file pathes and reporter codes
        dir_series = pd.Series(os.listdir(sub_path))
        avail_reporter_code_list=dir_series.str.split(".",expand=True)[0]

        # read in the first country
        df=pd.read_csv(sub_path+"/"+dir_series.iloc[0])[["commodity_code","dollar_value"]]
        df.columns=["commodity_code",avail_reporter_code_list.iloc[0]]

        # append all other countries to df
        for nxt in range(1,len(avail_reporter_code_list)):
            nxt_df=pd.read_csv(sub_path+"/"+dir_series.iloc[nxt])[["commodity_code","dollar_value"]]
            nxt_df.columns=["commodity_code",avail_reporter_code_list.iloc[nxt]]
            df=df.merge(nxt_df,left_on='commodity_code', right_on='commodity_code',how='outer')
        print(year)
        df.fillna(0,inplace=True)
        df.to_csv(sub_path+"_mega.csv")
    return 200


In [ ]:
sub_path

NameError: ignored

In [ ]:
# Function to add a service column to the mega csv and save to local
def get_service(year=2021):
    SERVICE_URL="https://comtrade.un.org/api/get?max=50000&type=S&freq=A&px=EB02&ps="+str(year)+"&r=all&p=0&rg=2&cc=200&uitoken=<COMTRADE_UITOKEN>&fmt=csv"
    with requests.Session() as s:
        download = s.get(SERVICE_URL)
        decoded_content = download.content.decode('utf-8')
        cr = csv.reader(decoded_content.splitlines(), delimiter=',')
        my_list = list(cr)
        df=pd.DataFrame(np.array(my_list))
        df=df.iloc[1:,[1,8,20]]
        df.columns=["year","reporter_code","dollar_value"]
    return df

# Save all countries' export data individually to the PATH folder

for reporter_code in reporter_code_list:
    for year in year_list:
        try:
            get_export(reporter_code,year)
            print(".",end="")
        except KeyboardInterrupt:
            raise
        except:
            print("X",end="")
            pass

In [ ]:

index=np.where(reporter_code_list==784)[0][0]
print(index)
for reporter_code in reporter_code_list[index:]:
    for year in year_list:
      if os.path.exists(PATH+str(year)+"/"+str(reporter_code)+".csv")==False:
        time.sleep(10)
        try:
            get_export(reporter_code,year)
            print(".",end="")
        except KeyboardInterrupt:
            raise
        except:
            print("X",end="")
            pass
      else:
        print('exists')


In [ ]:
get_export(156,2021)

# Merge into mega csvs

merge_all(PATH,year_list)

In [ ]:
merge_all(PATH,year_list)

drive/My Drive/ECI Research/Fall 2022/Data/raw_data/2019


FileNotFoundError: ignored

# Add Service data to the mega dataset

In [ ]:
get_service(year=2021)

SSLError: ignored